In [2]:
import os
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt
import pandas as pd
from surface_stats import cal_volume, get_surface_area

In [3]:
df = pd.read_csv("/mnt/sda/public/00_Projects/2512_AutoRecon/results/gfy_results/surface_stats_v2.csv")
wm_area_l_list = df["wm_area_l_list"].tolist()
wm_area_r_list = df["wm_area_r_list"].tolist()
wm_vol_l_list = df["wm_vol_l_list"].tolist()
wm_vol_r_list = df["wm_vol_r_list"].tolist()
cortex_area_l_list = df["cortex_area_l_list"].tolist()
cortex_area_r_list = df["cortex_area_r_list"].tolist()
cortex_vol_l_list = df["cortex_vol_l_list"].tolist()
cortex_vol_r_list = df["cortex_vol_r_list"].tolist()
left_hemispheres_vol = df["left_hemispheres_vol"].tolist()
right_hemispheres_vol = df["right_hemispheres_vol"].tolist()



In [4]:
outlier_idx =  17
print("删除异常值前的数据点个数:")
print(f"左白质：{len(wm_area_l_list)}")
print(f"右白质：{len(wm_area_r_list)}")
print(f"左皮层：{len(cortex_area_l_list)}")
print(f"右皮层：{len(cortex_area_r_list)}")
del wm_area_l_list[outlier_idx]
del wm_vol_l_list[outlier_idx]
del wm_area_r_list[outlier_idx]
del wm_vol_r_list[outlier_idx]
del cortex_area_l_list[outlier_idx]
del cortex_vol_l_list[outlier_idx]
del cortex_area_r_list[outlier_idx]
del cortex_vol_r_list[outlier_idx]
print("删除异常值后的数据点个数:")
print(f"左白质：{len(wm_area_l_list)}")
print(f"右白质：{len(wm_area_r_list)}")
print(f"左皮层：{len(cortex_area_l_list)}")
print(f"右皮层：{len(cortex_area_r_list)}")
del left_hemispheres_vol[outlier_idx]
del right_hemispheres_vol[outlier_idx]

删除异常值前的数据点个数:
左白质：100
右白质：100
左皮层：100
右皮层：100
删除异常值后的数据点个数:
左白质：99
右白质：99
左皮层：99
右皮层：99


In [5]:
def load_new_point(surface_file_dir, seg_dir):
    wm_area_l = get_surface_area(surface_file_dir,"lh.smoothwm.nofix")
    wm_area_r = get_surface_area(surface_file_dir,"rh.smoothwm.nofix")
    if seg_dir is not None:
        pass
    else:
        wm_vol_count_l, wm_vol_ml_l = cal_volume(f"{surface_file_dir}/seg_all.nii.gz", low_threshold=0.1, up_threshold=1.1)
        wm_vol_count_r, wm_vol_ml_r = cal_volume(f"{surface_file_dir}/seg_all.nii.gz", low_threshold=1.1, up_threshold=2.1)

    cortex_area_l = get_surface_area(surface_file_dir,"lh.smoothcortex.nofix")
    cortex_area_r = get_surface_area(surface_file_dir,"rh.smoothcortex.nofix")
    mass_vol_count_l, mass_vol_ml_l = cal_volume(f"{surface_file_dir}/cortex.nii.gz", low_threshold=2.1, up_threshold=3.1)
    mass_vol_count_r, mass_vol_ml_r = cal_volume(f"{surface_file_dir}/cortex.nii.gz", low_threshold=3.1, up_threshold=4.1)

    # cortex_vol_count_l, cortex_vol_ml_l = (mass_vol_count_l-wm_vol_count_l, mass_vol_ml_l-wm_vol_ml_l)
    # cortex_vol_count_r, cortex_vol_ml_r = (mass_vol_count_r-wm_vol_count_r, mass_vol_ml_r-wm_vol_ml_r)
    new_point = {
        "wm_area_l":wm_area_l,
        "wm_area_r":wm_area_r,
        "wm_vol_ml_l": wm_vol_ml_l,
        "wm_vol_ml_r": wm_vol_ml_r,
        "cortex_area_l":cortex_area_l,
        "cortex_area_r":cortex_area_r,
        "mass_vol_ml_l": mass_vol_ml_l,
        "mass_vol_ml_r": mass_vol_ml_r
    }
    return new_point

In [6]:
def plot_and_save(new_point, save_dir, id_name):
    plt.figure(figsize=(12, 10))
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False

    # 子图1：左半球白质
    plt.subplot(2, 2, 1)
    plt.scatter(wm_area_l_list, [x * 1000 for x in wm_vol_l_list], color='skyblue', s=60, alpha=0.8, label='Left WM')
    plt.scatter(
        new_point["wm_area_l"], new_point["wm_vol_ml_l"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}'
    )
    # 线性拟合
    z_l_wm = np.polyfit(wm_area_l_list, [x * 1000 for x in wm_vol_l_list], 1)
    p_l_wm = np.poly1d(z_l_wm)
    plt.plot(wm_area_l_list, p_l_wm(wm_area_l_list), "r--", alpha=0.6, label=f'Fit: y={z_l_wm[0]:.5f}x + {z_l_wm[1]:.2f}')
    plt.title('Left White Matter (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图2：右半球白质
    plt.subplot(2, 2, 2)
    plt.scatter(wm_area_r_list, [x * 1000 for x in wm_vol_r_list], color='lightcoral', s=60, alpha=0.8, label='Right WM')
    plt.scatter(
        new_point["wm_area_r"], new_point["wm_vol_ml_r"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}'
    )
    # 线性拟合
    z_r_wm = np.polyfit(wm_area_r_list, [x * 1000 for x in wm_vol_r_list], 1)
    p_r_wm = np.poly1d(z_r_wm)
    plt.plot(wm_area_r_list, p_r_wm(wm_area_r_list), "b--", alpha=0.6, label=f'Fit: y={z_r_wm[0]:.5f}x + {z_r_wm[1]:.2f}')
    plt.title('Right White Matter (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图3：左半球皮层
    plt.subplot(2, 2, 3)
    plt.scatter(cortex_area_l_list, [x * 1000 for x in left_hemispheres_vol], color='mediumseagreen', s=60, alpha=0.8, label='Left Cortex')
    plt.scatter(
        new_point["cortex_area_l"], new_point["mass_vol_ml_l"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}'
    )
    # 线性拟合
    z_l_cortex = np.polyfit(cortex_area_l_list, [x * 1000 for x in left_hemispheres_vol], 1)
    p_l_cortex = np.poly1d(z_l_cortex)
    plt.plot(cortex_area_l_list, p_l_cortex(cortex_area_l_list), "r--", alpha=0.6, label=f'Fit: y={z_l_cortex[0]:.5f}x + {z_l_cortex[1]:.2f}')
    plt.title('Left Cortex (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图4：右半球皮层
    plt.subplot(2, 2, 4)
    plt.scatter(cortex_area_r_list, [x * 1000 for x in right_hemispheres_vol], color='orange', s=60, alpha=0.8, label='Right Cortex')
    plt.scatter(
        new_point["cortex_area_r"], new_point["mass_vol_ml_r"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}'
    )
    # 线性拟合
    z_r_cortex = np.polyfit(cortex_area_r_list, [x * 1000 for x in right_hemispheres_vol], 1)
    p_r_cortex = np.poly1d(z_r_cortex)
    plt.plot(cortex_area_r_list, p_r_cortex(cortex_area_r_list), "b--", alpha=0.6, label=f'Fit: y={z_r_cortex[0]:.5f}x + {z_r_cortex[1]:.2f}')
    plt.title('Right Cortex (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 整体布局
    plt.suptitle('Brain Structure: Surface Area (X) vs Volume (Y)', fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # 保存图片
    save_path = f"{save_dir}/Subj_{id_name}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"带拟合曲线的图片已保存至：{save_path}")

In [7]:
def same_plot(new_point, save_dir, id_name):
    plt.figure(figsize=(12, 10))
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False

    # 子图1：左半球白质
    plt.subplot(2, 2, 1)
    plt.scatter(wm_area_l_list, [x * 1000 for x in wm_vol_l_list], color='skyblue', s=60, alpha=0.8, label='Left WM')
    plt.scatter(
        new_point["wm_area_l"], new_point["wm_vol_ml_l"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'left hemisphere'
    )
    plt.scatter(
        new_point["wm_area_r"], new_point["wm_vol_ml_r"]*1000,
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'right hemisphere'
    )
    # 线性拟合
    z_l_wm = np.polyfit(wm_area_l_list, [x * 1000 for x in wm_vol_l_list], 1)
    p_l_wm = np.poly1d(z_l_wm)
    plt.plot(wm_area_l_list, p_l_wm(wm_area_l_list), "r--", alpha=0.6, label=f'Fit: y={z_l_wm[0]:.5f}x + {z_l_wm[1]:.2f}')
    plt.title('Left White Matter (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图2：右半球白质
    plt.subplot(2, 2, 2)
    plt.scatter(wm_area_r_list, [x * 1000 for x in wm_vol_r_list], color='lightcoral', s=60, alpha=0.8, label='Right WM')
    plt.scatter(
        new_point["wm_area_l"], new_point["wm_vol_ml_l"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'left hemisphere'
    )
    plt.scatter(
        new_point["wm_area_r"], new_point["wm_vol_ml_r"]*1000,
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'right hemisphere'
    )
    # 线性拟合
    z_r_wm = np.polyfit(wm_area_r_list, [x * 1000 for x in wm_vol_r_list], 1)
    p_r_wm = np.poly1d(z_r_wm)
    plt.plot(wm_area_r_list, p_r_wm(wm_area_r_list), "b--", alpha=0.6, label=f'Fit: y={z_r_wm[0]:.5f}x + {z_r_wm[1]:.2f}')
    plt.title('Right White Matter (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图3：左半球皮层
    plt.subplot(2, 2, 3)
    plt.scatter(cortex_area_l_list, [x * 1000 for x in left_hemispheres_vol], color='mediumseagreen', s=60, alpha=0.8, label='Left Cortex')
    plt.scatter(
        new_point["cortex_area_l"], new_point["mass_vol_ml_l"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'left hemisphere'
    )
    plt.scatter(
        new_point["cortex_area_r"], new_point["mass_vol_ml_r"]*1000,
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'right hemisphere'
    )
    # 线性拟合
    z_l_cortex = np.polyfit(cortex_area_l_list, [x * 1000 for x in left_hemispheres_vol], 1)
    p_l_cortex = np.poly1d(z_l_cortex)
    plt.plot(cortex_area_l_list, p_l_cortex(cortex_area_l_list), "r--", alpha=0.6, label=f'Fit: y={z_l_cortex[0]:.5f}x + {z_l_cortex[1]:.2f}')
    plt.title('Left Cortex (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图4：右半球皮层
    plt.subplot(2, 2, 4)
    plt.scatter(cortex_area_r_list, [x * 1000 for x in right_hemispheres_vol], color='orange', s=60, alpha=0.8, label='Right Cortex')
    plt.scatter(
        new_point["cortex_area_l"], new_point["mass_vol_ml_l"]*1000,
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'left hemisphere'
    )
    plt.scatter(
        new_point["cortex_area_r"], new_point["mass_vol_ml_r"]*1000,
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'right '
    )
    # 线性拟合
    z_r_cortex = np.polyfit(cortex_area_r_list, [x * 1000 for x in right_hemispheres_vol], 1)
    p_r_cortex = np.poly1d(z_r_cortex)
    plt.plot(cortex_area_r_list, p_r_cortex(cortex_area_r_list), "b--", alpha=0.6, label=f'Fit: y={z_r_cortex[0]:.5f}x + {z_r_cortex[1]:.2f}')
    plt.title('Right Cortex (Area vs Volume)', fontsize=12)
    plt.xlabel('Surface Area (mm²)', fontsize=10)
    plt.ylabel('Volume (mm³)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 整体布局
    plt.suptitle('Brain Structure: Surface Area (X) vs Volume (Y)', fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # 保存图片
    save_path = f"{save_dir}/Subj_{id_name}_both.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"带拟合曲线的图片已保存至：{save_path}")

In [8]:
def same_plot_chage_order(new_point, save_dir, id_name):
    plt.figure(figsize=(12, 10))
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False

    # 子图1：左半球白质 【互换：X=体积 Volume，Y=面积 Area】
    plt.subplot(2, 2, 1)
    wm_vol_l_mm3 = [x * 1000 for x in wm_vol_l_list]
    plt.scatter(wm_vol_l_mm3, wm_area_l_list, color='skyblue', s=60, alpha=0.8, label='Left WM')
    plt.scatter(
        new_point["wm_vol_ml_l"]*1000, new_point["wm_area_l"],
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}-left'
    )
    plt.scatter(
        new_point["wm_vol_ml_r"]*1000, new_point["wm_area_r"],
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'Subject-{id_name}-right'
    )
    # 线性拟合：互换x/y数据做拟合
    z_l_wm = np.polyfit(wm_vol_l_mm3, wm_area_l_list, 1)
    p_l_wm = np.poly1d(z_l_wm)
    plt.plot(wm_vol_l_mm3, p_l_wm(wm_vol_l_mm3), "r--", alpha=0.6, label=f'Fit: y={z_l_wm[0]:.5f}x + {z_l_wm[1]:.2f}')
    plt.title('Left White Matter (Volume vs Area)', fontsize=12)
    plt.xlabel('Volume (mm³)', fontsize=10)
    plt.ylabel('Surface Area (mm²)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图2：右半球白质 【互换：X=体积 Volume，Y=面积 Area】
    plt.subplot(2, 2, 2)
    wm_vol_r_mm3 = [x * 1000 for x in wm_vol_r_list]
    plt.scatter(wm_vol_r_mm3, wm_area_r_list, color='lightcoral', s=60, alpha=0.8, label='Right WM')
    plt.scatter(
        new_point["wm_vol_ml_l"]*1000, new_point["wm_area_l"],
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}-left'
    )
    plt.scatter(
        new_point["wm_vol_ml_r"]*1000, new_point["wm_area_r"],
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'Subject-{id_name}-right'
    )
    # 线性拟合：互换x/y数据做拟合
    z_r_wm = np.polyfit(wm_vol_r_mm3, wm_area_r_list, 1)
    p_r_wm = np.poly1d(z_r_wm)
    plt.plot(wm_vol_r_mm3, p_r_wm(wm_vol_r_mm3), "b--", alpha=0.6, label=f'Fit: y={z_r_wm[0]:.5f}x + {z_r_wm[1]:.2f}')
    plt.title('Right White Matter (Volume vs Area)', fontsize=12)
    plt.xlabel('Volume (mm³)', fontsize=10)
    plt.ylabel('Surface Area (mm²)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图3：左半球皮层 【互换：X=体积 Volume，Y=面积 Area】
    plt.subplot(2, 2, 3)
    left_cortex_vol_mm3 = [x * 1000 for x in left_hemispheres_vol]
    plt.scatter(left_cortex_vol_mm3, cortex_area_l_list, color='mediumseagreen', s=60, alpha=0.8, label='Left Cortex')
    plt.scatter(
        new_point["mass_vol_ml_l"]*1000, new_point["cortex_area_l"],
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}-left'
    )
    plt.scatter(
        new_point["mass_vol_ml_r"]*1000, new_point["cortex_area_r"],
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'Subject-{id_name}-right'
    )
    # 线性拟合：互换x/y数据做拟合
    z_l_cortex = np.polyfit(left_cortex_vol_mm3, cortex_area_l_list, 1)
    p_l_cortex = np.poly1d(z_l_cortex)
    plt.plot(left_cortex_vol_mm3, p_l_cortex(left_cortex_vol_mm3), "r--", alpha=0.6, label=f'Fit: y={z_l_cortex[0]:.5f}x + {z_l_cortex[1]:.2f}')
    plt.title('Left Cortex (Volume vs Area)', fontsize=12)
    plt.xlabel('Volume (mm³)', fontsize=10)
    plt.ylabel('Surface Area (mm²)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 子图4：右半球皮层 【互换：X=体积 Volume，Y=面积 Area】
    plt.subplot(2, 2, 4)
    right_cortex_vol_mm3 = [x * 1000 for x in right_hemispheres_vol]
    plt.scatter(right_cortex_vol_mm3, cortex_area_r_list, color='orange', s=60, alpha=0.8, label='Right Cortex')
    plt.scatter(
        new_point["mass_vol_ml_l"]*1000, new_point["cortex_area_l"],
        color='red', s=200, marker='*', alpha=1.0, 
        label=f'Subject-{id_name}-left'
    )
    plt.scatter(
        new_point["mass_vol_ml_r"]*1000, new_point["cortex_area_r"],
        color='blue', s=200, marker='*', alpha=0.8, 
        label=f'Subject-{id_name}-right'
    )
    # 线性拟合：互换x/y数据做拟合
    z_r_cortex = np.polyfit(right_cortex_vol_mm3, cortex_area_r_list, 1)
    p_r_cortex = np.poly1d(z_r_cortex)
    plt.plot(right_cortex_vol_mm3, p_r_cortex(right_cortex_vol_mm3), "b--", alpha=0.6, label=f'Fit: y={z_r_cortex[0]:.5f}x + {z_r_cortex[1]:.2f}')
    plt.title('Right Cortex (Volume vs Area)', fontsize=12)
    plt.xlabel('Volume (mm³)', fontsize=10)
    plt.ylabel('Surface Area (mm²)', fontsize=10)
    plt.grid(alpha=0.3)
    plt.legend()

    # 整体布局
    plt.suptitle('Brain Structure: Volume (X) vs Surface Area (Y)', fontsize=16, y=0.98)
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # 保存图片
    save_path = f"{save_dir}/Subj_{id_name}_both.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"带拟合曲线的图片已保存至：{save_path}")

In [9]:
patch_dir = "/mnt/sda/public/00_Projects/2512_AutoRecon/data/surface"
save_dir = "/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis"
# sub_dir_subjet_list = ["03", "06", "07", "08"]
# sub_dir_subjet_list = ["03", "06", "07", "08", "01_cqq", "00_zwy"]
# sub_dir_subjet_list = ["02", "09", "10"]
# sub_dir_subjet_list=["11"]
sub_dir_subjet_list=["12", "13"]
# sub_dir_subjet_list=["00_zwy"]
for sub_dir in sub_dir_subjet_list:
    new_data  = load_new_point(f"{patch_dir}/{sub_dir}", None)
    plot_and_save(new_data, save_dir, sub_dir)
    same_plot(new_data, save_dir, sub_dir)
    # same_plot_chage_order(new_data, save_dir, sub_dir)

带拟合曲线的图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_12.png
带拟合曲线的图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_12_both.png
带拟合曲线的图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_13.png
带拟合曲线的图片已保存至：/mnt/sda/public/00_Projects/2512_AutoRecon/results/njgl_analysis/Subj_13_both.png
